# Train Network Metrics Dashboard

In this notebook, you will analyze train network performance using time-series metrics:

1. Plot waiting passenger counts over time for each station
2. Track trains deployed throughout the simulation
3. Calculate throughput (passengers boarded per time interval)
4. Compare actual vs expected capacity
5. Identify bottlenecks and optimization opportunities

This provides quantitative analysis of the simulation behavior.

## Step 1: Setup and Configuration

Load configuration and prepare data collection structures.

In [ ]:
from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector
from simulated_city.agents import (
    Station,
    SimulationState,
    TrainAgent,
    PassengerSourceAgent,
    SensorAgent,
    ControlCenterAgent,
    DispatcherAgent,
)
from datetime import datetime, timedelta
import time
import json
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Load configuration
config = load_config()

print("✓ Configuration loaded")
print(f"  Dispatcher threshold: {config.train_network.dispatcher.waiting_threshold} passengers")
print(f"  Base train capacity: {config.train_network.train_config.capacity}")

## Step 2: Initialize Data Collection

Create structures to store time-series data during simulation.

In [ ]:
# Data collection structures
metrics = {
    "timestamps": [],
    "waiting_counts": defaultdict(list),  # per station
    "trains_deployed": [],
    "passengers_boarded": [],
    "passengers_alighted": [],
    "total_waiting": [],
}

# Simulation parameters
SIMULATION_DURATION_MINUTES = 20
SAMPLE_INTERVAL_SECONDS = 10  # Collect metrics every 10 seconds

print(f"Simulation duration: {SIMULATION_DURATION_MINUTES} minutes")
print(f"Sample interval: {SAMPLE_INTERVAL_SECONDS} seconds")
print(f"Expected samples: {(SIMULATION_DURATION_MINUTES * 60) // SAMPLE_INTERVAL_SECONDS}")

## Step 3: Setup Simulation State and Agents

Initialize the simulation components.

In [ ]:
# Connect to MQTT
connector = MqttConnector(config.mqtt, client_id_suffix="dashboard")
connector.connect()

if connector.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
else:
    print("✗ Failed to connect to MQTT broker")
    raise Exception("MQTT connection required for dashboard")

from simulated_city.mqtt import MqttPublisher
publisher = MqttPublisher(connector)

# Initialize simulation state
sim_state = SimulationState(current_time=datetime.now())

# Convert config to Station objects
route = []
for station_config in config.train_network.route:
    station = Station(
        name=station_config.name,
        station_type=station_config.type,
        location_lat=station_config.location.lat,
        location_lon=station_config.location.lon,
        exit_percentage=station_config.exit_percentage,
    )
    route.append(station)

entry_stations = [s for s in route if s.station_type == "entry"]

print(f"\n✓ Simulation state initialized")
print(f"  Entry stations: {len(entry_stations)}")
print(f"  Exit stations: {len([s for s in route if s.station_type == 'exit'])}")

In [ ]:
# Create passenger source agents
passenger_sources = []
for station in entry_stations:
    source = PassengerSourceAgent(
        station=station,
        passenger_flow_config=config.train_network.passenger_flow,
        simulation_state=sim_state,
    )
    passenger_sources.append(source)

# Create sensor agents
sensor_agents = []
for station in entry_stations:
    sensor = SensorAgent(
        station=station,
        mqtt_publisher=publisher,
        mqtt_base_topic=config.train_network.mqtt_base_topic,
    )
    sensor_agents.append(sensor)

print(f"✓ Created {len(passenger_sources)} passenger source agents")
print(f"✓ Created {len(sensor_agents)} sensor agents")

## Step 4: Run Simulation with Metrics Collection

Execute the simulation and collect data for visualization.

In [ ]:
print("Starting simulation...\n")

start_time = datetime.now()
end_time = start_time + timedelta(minutes=SIMULATION_DURATION_MINUTES)
current_time = start_time

# Track cumulative metrics
total_boarded = 0
total_alighted = 0
trains_in_service = 3  # Starting with base trains

sample_count = 0

while current_time < end_time:
    # Update simulation time
    sim_state.current_time = current_time
    
    # Generate passengers
    for source in passenger_sources:
        # Generate passengers for this 10-second interval
        # Scale from per-10min rate to per-10sec
        rate_per_10sec = source.get_current_rate(current_time) / 60
        passengers_to_add = int(rate_per_10sec)
        
        if passengers_to_add > 0:
            source.generate_passengers(count=passengers_to_add)
    
    # Collect metrics
    metrics["timestamps"].append(current_time)
    
    total_waiting_now = 0
    for station in entry_stations:
        queue = sim_state.get_station_queue(station.name)
        count = queue.count
        metrics["waiting_counts"][station.name].append(count)
        total_waiting_now += count
    
    metrics["total_waiting"].append(total_waiting_now)
    metrics["trains_deployed"].append(trains_in_service)
    
    # Simulate boarding/alighting (simplified)
    # Each train can handle ~60 passengers per 10-second stop
    passengers_per_interval = trains_in_service * 60
    boarded_this_interval = min(total_waiting_now, passengers_per_interval)
    total_boarded += boarded_this_interval
    
    # Estimate alighting (about 70% eventually disembark)
    alighted_this_interval = int(boarded_this_interval * 0.7)
    total_alighted += alighted_this_interval
    
    metrics["passengers_boarded"].append(total_boarded)
    metrics["passengers_alighted"].append(total_alighted)
    
    # Every 2 minutes, check if we need extra trains
    if sample_count % 12 == 0:  # 12 samples = 2 minutes
        if total_waiting_now > config.train_network.dispatcher.waiting_threshold:
            trains_in_service += 1
            print(f"[{current_time.strftime('%H:%M:%S')}] Extra train deployed (total: {trains_in_service})")
    
    # Progress update
    if sample_count % 6 == 0:  # Every minute
        print(f"[{current_time.strftime('%H:%M:%S')}] Waiting: {total_waiting_now}, Trains: {trains_in_service}, Boarded: {total_boarded}")
    
    sample_count += 1
    current_time += timedelta(seconds=SAMPLE_INTERVAL_SECONDS)
    time.sleep(0.1)  # Small delay for readability

print(f"\n✓ Simulation complete")
print(f"  Samples collected: {len(metrics['timestamps'])}")
print(f"  Total passengers boarded: {total_boarded}")
print(f"  Total passengers alighted: {total_alighted}")

## Step 5: Plot Waiting Passengers Over Time

Visualize queue sizes at each entry station.

In [ ]:
# Create figure with subplots
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Waiting passengers per station
ax1 = axes[0]
for station in entry_stations:
    ax1.plot(
        metrics["timestamps"],
        metrics["waiting_counts"][station.name],
        label=station.name,
        linewidth=2,
        marker='o',
        markersize=3
    )

# Add threshold line
threshold = config.train_network.dispatcher.waiting_threshold
ax1.axhline(y=threshold, color='r', linestyle='--', linewidth=2, label=f'Threshold ({threshold})')

ax1.set_xlabel("Time", fontsize=12)
ax1.set_ylabel("Waiting Passengers", fontsize=12)
ax1.set_title("Passenger Queues at Entry Stations Over Time", fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

# Plot 2: Total waiting across all stations
ax2 = axes[1]
ax2.plot(
    metrics["timestamps"],
    metrics["total_waiting"],
    label="Total Waiting",
    linewidth=2,
    color='#2196F3',
    marker='o',
    markersize=3
)
ax2.fill_between(metrics["timestamps"], metrics["total_waiting"], alpha=0.3, color='#2196F3')

ax2.set_xlabel("Time", fontsize=12)
ax2.set_ylabel("Total Passengers", fontsize=12)
ax2.set_title("Total Network Queue Size", fontsize=14, fontweight='bold')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

plt.tight_layout()
plt.show()

print("✓ Queue visualization complete")

## Step 6: Plot Train Deployments

Show how many trains are in service over time.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 5))

ax.plot(
    metrics["timestamps"],
    metrics["trains_deployed"],
    label="Trains in Service",
    linewidth=2,
    color='#4CAF50',
    marker='s',
    markersize=4,
    drawstyle='steps-post'
)
ax.fill_between(
    metrics["timestamps"],
    metrics["trains_deployed"],
    alpha=0.3,
    color='#4CAF50',
    step='post'
)

ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("Number of Trains", fontsize=12)
ax.set_title("Train Deployments Over Time", fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

print(f"✓ Train deployment visualization complete")
print(f"  Peak trains: {max(metrics['trains_deployed'])}")
print(f"  Average trains: {sum(metrics['trains_deployed']) / len(metrics['trains_deployed']):.1f}")

## Step 7: Plot Throughput Metrics

Visualize cumulative passengers boarded and alighted.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

ax.plot(
    metrics["timestamps"],
    metrics["passengers_boarded"],
    label="Cumulative Boarded",
    linewidth=2,
    color='#2196F3',
    marker='o',
    markersize=3
)
ax.plot(
    metrics["timestamps"],
    metrics["passengers_alighted"],
    label="Cumulative Alighted",
    linewidth=2,
    color='#FF9800',
    marker='s',
    markersize=3
)

ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("Passengers", fontsize=12)
ax.set_title("Cumulative Passenger Throughput", fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

plt.tight_layout()
plt.show()

print("✓ Throughput visualization complete")
print(f"  Final boarded: {metrics['passengers_boarded'][-1]}")
print(f"  Final alighted: {metrics['passengers_alighted'][-1]}")
print(f"  Still on trains: {metrics['passengers_boarded'][-1] - metrics['passengers_alighted'][-1]}")

## Step 8: Calculate Performance Metrics

Compute key performance indicators for the simulation.

In [ ]:
# Calculate KPIs
duration_seconds = (metrics["timestamps"][-1] - metrics["timestamps"][0]).total_seconds()
duration_minutes = duration_seconds / 60

avg_waiting = sum(metrics["total_waiting"]) / len(metrics["total_waiting"])
max_waiting = max(metrics["total_waiting"])
avg_trains = sum(metrics["trains_deployed"]) / len(metrics["trains_deployed"])

# Throughput (passengers per minute)
throughput_per_min = total_boarded / duration_minutes

# Expected capacity (based on train capacity and service frequency)
expected_capacity_per_min = avg_trains * config.train_network.train_config.capacity / 10

# Capacity utilization
capacity_utilization = (throughput_per_min / expected_capacity_per_min) * 100 if expected_capacity_per_min > 0 else 0

print("=" * 60)
print("PERFORMANCE METRICS")
print("=" * 60)
print(f"\nSimulation Duration: {duration_minutes:.1f} minutes")
print(f"\nPassenger Flow:")
print(f"  Total Boarded:     {total_boarded:,}")
print(f"  Total Alighted:    {total_alighted:,}")
print(f"  Throughput:        {throughput_per_min:.1f} passengers/minute")
print(f"\nQueue Statistics:")
print(f"  Average Waiting:   {avg_waiting:.1f} passengers")
print(f"  Peak Waiting:      {max_waiting} passengers")
print(f"  Threshold:         {config.train_network.dispatcher.waiting_threshold} passengers")
print(f"\nTrain Operations:")
print(f"  Average Trains:    {avg_trains:.1f}")
print(f"  Peak Trains:       {max(metrics['trains_deployed'])}")
print(f"  Base Trains:       {min(metrics['trains_deployed'])}")
print(f"\nCapacity Analysis:")
print(f"  Expected Capacity: {expected_capacity_per_min:.1f} passengers/min")
print(f"  Actual Throughput: {throughput_per_min:.1f} passengers/min")
print(f"  Utilization:       {capacity_utilization:.1f}%")

# Identify bottlenecks
print(f"\nBottleneck Analysis:")
for station in entry_stations:
    avg_queue = sum(metrics["waiting_counts"][station.name]) / len(metrics["waiting_counts"][station.name])
    max_queue = max(metrics["waiting_counts"][station.name])
    print(f"  {station.name}:")
    print(f"    Average Queue: {avg_queue:.1f}")
    print(f"    Peak Queue:    {max_queue}")
    if max_queue > threshold:
        print(f"    ⚠️  Exceeded threshold by {max_queue - threshold} passengers")

## Step 9: Combined Dashboard View

Create a comprehensive multi-panel dashboard.

In [ ]:
# Create comprehensive dashboard
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# Panel 1: Waiting passengers per station
ax1 = fig.add_subplot(gs[0, :])
for station in entry_stations:
    ax1.plot(metrics["timestamps"], metrics["waiting_counts"][station.name], label=station.name, linewidth=2)
ax1.axhline(y=threshold, color='r', linestyle='--', linewidth=2, alpha=0.7)
ax1.set_ylabel("Waiting Passengers")
ax1.set_title("Queue Sizes by Station", fontweight='bold')
ax1.legend(loc='upper left', ncol=3)
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

# Panel 2: Total waiting
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(metrics["timestamps"], metrics["total_waiting"], linewidth=2, color='#2196F3')
ax2.fill_between(metrics["timestamps"], metrics["total_waiting"], alpha=0.3, color='#2196F3')
ax2.set_ylabel("Total Passengers")
ax2.set_title("Total Network Queue", fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

# Panel 3: Trains deployed
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(metrics["timestamps"], metrics["trains_deployed"], linewidth=2, color='#4CAF50', drawstyle='steps-post')
ax3.fill_between(metrics["timestamps"], metrics["trains_deployed"], alpha=0.3, color='#4CAF50', step='post')
ax3.set_ylabel("Trains in Service")
ax3.set_title("Train Deployments", fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

# Panel 4: Throughput
ax4 = fig.add_subplot(gs[2, 0])
ax4.plot(metrics["timestamps"], metrics["passengers_boarded"], linewidth=2, label="Boarded", color='#2196F3')
ax4.plot(metrics["timestamps"], metrics["passengers_alighted"], linewidth=2, label="Alighted", color='#FF9800')
ax4.set_ylabel("Cumulative Passengers")
ax4.set_xlabel("Time")
ax4.set_title("Passenger Throughput", fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

# Panel 5: Summary stats (text)
ax5 = fig.add_subplot(gs[2, 1])
ax5.axis('off')
summary_text = f"""
KEY METRICS

Total Boarded: {total_boarded:,}
Total Alighted: {total_alighted:,}

Avg Queue: {avg_waiting:.0f} passengers
Peak Queue: {max_waiting} passengers

Avg Trains: {avg_trains:.1f}
Peak Trains: {max(metrics['trains_deployed'])}

Throughput: {throughput_per_min:.1f} pax/min
Utilization: {capacity_utilization:.1f}%
"""
ax5.text(0.1, 0.5, summary_text, fontsize=11, family='monospace', verticalalignment='center')

fig.suptitle(f"Train Network Performance Dashboard ({duration_minutes:.0f} min simulation)", 
             fontsize=16, fontweight='bold', y=0.995)

plt.show()

print("✓ Dashboard complete")

## Step 10: Export Metrics Data

Save the collected metrics for further analysis.

In [ ]:
# Export to CSV-like format
export_data = {
    "simulation": {
        "start_time": start_time.isoformat(),
        "end_time": end_time.isoformat(),
        "duration_minutes": duration_minutes,
        "sample_interval_seconds": SAMPLE_INTERVAL_SECONDS,
    },
    "configuration": {
        "threshold": config.train_network.dispatcher.waiting_threshold,
        "train_capacity": config.train_network.train_config.capacity,
        "entry_stations": len(entry_stations),
    },
    "kpis": {
        "total_boarded": total_boarded,
        "total_alighted": total_alighted,
        "avg_waiting": avg_waiting,
        "max_waiting": max_waiting,
        "avg_trains": avg_trains,
        "peak_trains": max(metrics["trains_deployed"]),
        "throughput_per_min": throughput_per_min,
        "capacity_utilization_percent": capacity_utilization,
    },
    "time_series": {
        "timestamps": [t.isoformat() for t in metrics["timestamps"]],
        "total_waiting": metrics["total_waiting"],
        "trains_deployed": metrics["trains_deployed"],
        "passengers_boarded": metrics["passengers_boarded"],
        "passengers_alighted": metrics["passengers_alighted"],
    }
}

print("✓ Metrics exported")
print(f"  Total data points: {len(metrics['timestamps'])}")

# Optionally save to file
# import json
# with open('train_metrics.json', 'w') as f:
#     json.dump(export_data, f, indent=2)
# print("✓ Saved to train_metrics.json")

## Cleanup

In [ ]:
# Disconnect from MQTT
if connector.client and connector.client.is_connected():
    connector.disconnect()
    print("✓ Disconnected from MQTT broker")

## Exercise: Performance Optimization

Try these experiments to improve network performance:

1. **Adjust threshold**: Lower the dispatcher threshold in `config.yaml` from 250 to 200 and observe:
   - How quickly extra trains are deployed
   - Impact on average queue sizes
   - Peak waiting passenger count

2. **Increase capacity**: Change train capacity from 180 to 220 and measure:
   - Throughput improvement
   - Capacity utilization change
   - Whether fewer trains are needed

3. **Compare peak hours**: Modify passenger generation rates to simulate different rush hour patterns

4. **Multi-scenario analysis**: Run the simulation multiple times with different configs and compare KPIs

Example for experiment 1:
```python
# In config.yaml, change:
# dispatcher:
#   waiting_threshold: 200  # Was 250

# Re-run the simulation and compare:
# - Average queue sizes
# - Number of extra trains deployed
# - Capacity utilization
```

Example for experiment 2:
```python
# Track multiple scenarios
scenarios = []
for capacity in [150, 180, 210, 240]:
    # Update config
    # Run simulation
    # Store KPIs
    scenarios.append({
        'capacity': capacity,
        'avg_waiting': avg_waiting,
        'utilization': capacity_utilization
    })

# Plot comparison
plt.bar([s['capacity'] for s in scenarios], [s['avg_waiting'] for s in scenarios])
plt.xlabel('Train Capacity')
plt.ylabel('Average Queue Size')
plt.title('Impact of Train Capacity on Queue Sizes')
plt.show()
```

## Next Steps

You've completed the train network simulation implementation! Key takeaways:

- **Agent-based modeling**: Multiple autonomous agents (trains, sensors, control center) interact to create emergent system behavior
- **MQTT communication**: Distributed publish-subscribe messaging enables loose coupling between components
- **Threshold-based control**: Simple rules (dispatch when waiting > threshold) can manage complex systems
- **Performance metrics**: Time-series analysis reveals bottlenecks and optimization opportunities

Consider extending the simulation:
- Add route delays or breakdowns
- Implement weather impacts on capacity
- Create different station types (transfer stations, express stops)
- Add passenger destination preferences beyond entry/exit